<div dir="rtl">
<h1>کدام عددها واقعاً آموخته می‌شوند؟</h1>
<p>درس 50 از 76 · چند عدد می‌آموزیم و چرا آغاز تصادفی است؟ · <code dir="ltr">44-parameters</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-01/44-parameters.html">📖 بازگشت به همین درس</a></p>
<p>شمارش معماری را با Parameterهای واقعی بسنجید و Buffer را از وزن جدا کنید.</p><p>پیش‌نیاز: اندازهٔ Embedding، Linear، Layer Normalization و تعداد بلوک‌ها را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر فقط H عوض شود ولی C ثابت بماند، وزن‌های QKV بزرگ‌تر می‌شوند؟ آیا هر Tensor موجود در state_dict قابل آموزش است؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
config = ModelConfig(12,8,16,2,2,0.)
model = MiniGPT(config)
print('parameter tensors:',len(list(model.parameters())))
print('buffers:',[(name,tuple(value.shape)) for name,value in model.named_buffers()])
print('initial embedding std (small sample):',model.token_embedding.weight.detach().std().item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع parameter_formula(config) را از ابعاد Config بنویسید و یک int برگردانید. سهم دو جدول V*C، جدول موقعیت، L بلوک و Layer Normalization نهایی را جدا حساب کنید؛ در این تابع مدل نسازید و parameters را نشمارید.</p>
</div>

In [ ]:
def parameter_formula(config):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = parameter_formula(config)
    if result is None: return False
    assert result == 7104
    for cfg in (config,ModelConfig(9,5,12,3,1,0.),ModelConfig(15,7,8,4,3,0.)):
        actual = MiniGPT(cfg)
        assert parameter_formula(cfg) == sum(p.numel() for p in actual.parameters())
    assert parameter_formula(ModelConfig(12,8,16,4,2,0.)) == result
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط H را میان ۱، ۲ و ۴ تغییر دهید؛ اندازهٔ C و بقیهٔ Config ثابت‌اند. مقداردهی تصادفی مدل‌های تازه ربطی به شمار Parameterها ندارد.</p>
</div>

In [ ]:
for H in (1,2,4):
    candidate = MiniGPT(ModelConfig(12,8,16,H,2,0.))
    print('H, parameters:',H,sum(p.numel() for p in candidate.parameters()))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>کد خراب Buffer Causal Mask را هم وزن آموختنی شمرده است. تابع trainable_count(module) فقط Parameterهای requires_grad=True را بشمارد؛ ممکن است requires_grad بعضی وزن‌ها عمداً False شده باشد تا آموزش نبینند.</p>
</div>

In [ ]:
print('all state tensors:',sum(value.numel() for value in model.state_dict().values()))
print('parameters only:',sum(p.numel() for p in model.parameters()))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def trainable_count(module):
    # TODO
    return None

In [ ]:
def test_repair():
    result = trainable_count(model)
    if result is None: return False
    assert result == 7104
    model.token_embedding.weight.requires_grad_(False)
    try:
        assert trainable_count(model) == 7104-model.token_embedding.weight.numel()
    finally:
        model.token_embedding.weight.requires_grad_(True)
    assert trainable_count(model.final_norm) == 32
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>فرمول برای MiniGPT همین مخزن است: Head بدون Bias و بدون اشتراک وزن با Embedding، FFN با عرض 4C و دو Layer Normalization در هر بلوک. برای معماری دیگری باید فرمول را دوباره بسازید.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چه تفاوتی میان «Parameter مدل»، «Parameter قابل آموزش در این اجرا» و «همهٔ Tensorهای ذخیره‌شده» پیدا کردید؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-01/44-parameters.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/44-parameters.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>